### Dictionary

In [1]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.interpolate import BSpline, make_interp_spline

### Overview #2
Processes a data directory and organizes it into a dictionary for efficient iteration when creating box plots. Instead of iterating by subject ID in the CleanedData directory, this structure iterates directly by gesture.

### Dictionary 
* 12 Keys: Each key represents one of the 12 different gestures.
* Values: Each key contains a list with two sublists:
    * First sublist: Paths to all CSV files for freeform sessions except subject 1.
    * Second sublist: Paths to all CSV files for instructional sessions except subject 1.

### Variables
* cleaned_data_folder_path: Path to folder with all the cleaned data. 
    * Ensure that your CleanedData directory contains Sub folders -> Session Type folders -> 13 data files for each gesture. Should have this from running export_clean_files.py

In [2]:
cleaned_data_folder_path = os.path.join('..', 'CleanedData')
gesture_dict = {}

# Iterate through CleanedData directory to create a dictionary of gestures
# Each of the 12 gesture has two lists (freeform and instructional) containing paths to CSV data files.
for root, sub_folders, files in os.walk(cleaned_data_folder_path):
    sub_folders.sort()
    files.sort()
    
    for file in files:
        # Splits file name for gesture and session type identification
        file_split = file.split("_") 
        file_gesture, file_session_type, subject_id = file_split[3], file_split[2], file_split[5]

        # Skip unnecessary files: Box Select or Subject 1
        if file_gesture == 'BoxSelect' or subject_id == '1':
            continue

        # Skip the thank you/non-session files
        if file_session_type not in {'F', 'I'}:
            continue

        # Initialize gesture key in the dictionary if not already present
        if file_gesture not in gesture_dict:
            gesture_dict[file_gesture] = ([], [])  # [freeform_list, instructional_list]

        # Append file path to the appropriate session list
        if file_session_type == 'F':
            gesture_dict[file_gesture][0].append(os.path.join(root, file))
        else:
            gesture_dict[file_gesture][1].append(os.path.join(root, file))

### Overview #3
Bad stroke removal: The methods outlined below are used to remove strokes with "bad" shapes.

### Functions
* disect_cell: Determines which strokes to be removed based on cell value in the stroke removal file.
* remove_strokes: Remove unwanted strokes, such as accidental strokes or small dots, that were unintentionally created. 
* remove_extremities: Accounts for extremities or "hooks". These hooks refer to sharp curves typically formed at the start or end of a gesture, where participants often create abrupt movements after pulling or just before releasing the triggers. The function below utilizes the median absolute deviation (MAD) to detect and handle these extremities.

### Variables
* df_for_stroke_removal: path to the stroke removal csv file.

In [3]:
strokes2remove_path = os.path.join('..', 'PreprocessedDataML', 'stroke_shapes_to_remove.csv')
df_for_stroke_removal = pd.read_csv(strokes2remove_path)

def disect_cell(input):
    """
    Determines which strokes to remove based on cell value in the stroke removal file.

    Parameters
    -----
    input : str
        value in cell. Has the format of [3:5], [2], or [4:] where the number indicates the stroke and ':' indicates a range.

    Returns
    -----
    list_strokes_to_remove : str
        list of strokes to remove
    """

    i, list_strokes_to_remove = 0, []
    while i < len(input):
        if input[i] == '[':
            i += 1
            # Indicates a range of strokes to be removed
            if input[i+1] == ':':
                for j in range(int(input[i]), int(input[i+2])+1):
                    list_strokes_to_remove.append(j)
                i += 1
            # A single stroke is to be removed
            else:
                list_strokes_to_remove.append(int(input[i]))
                i += 1
        else:
            i += 1

    return list_strokes_to_remove

def remove_strokes(df, gesture_key_index, row_index):
    """
    Removes strokes by setting trigger pulls in that trial to 0.

    Parameters
    -----
    df : dataframe
        dataframe for a single sub session data file such as sub 6 session 3.
    gesture_key_index : int
        index of the gesture, which will be used to access the correct column in the stroke removal file.
    row_index : int
        index of the row that indicates which subject session. Will be used to access the correct row in the stroke removal file.
    """

    l_col_index, r_col_index = gesture_key_index * 2 + 1, gesture_key_index * 2 + 2
    value_in_left_cell = df_for_stroke_removal.iloc[row_index, l_col_index]
    value_in_right_cell = df_for_stroke_removal.iloc[row_index, r_col_index]

    # Determine which strokes to remove in that list
    l_strokes_to_remove = disect_cell(value_in_left_cell) if pd.notna(value_in_left_cell) else []
    r_strokes_to_remove = disect_cell(value_in_right_cell) if pd.notna(value_in_right_cell) else []

    # Change original df's trigger_pull_amount to 0 where strokes are to be removed
    for stroke_num in l_strokes_to_remove:
        df.loc[df['gesture_counter_UI'] == stroke_num, 'trigger_pull_amount_left'] = 0
        df.loc[df['gesture_counter_UI'] == stroke_num, 'trigger_pull_amount_right'] = 0
    for stroke_num in r_strokes_to_remove:
        df.loc[df['gesture_counter_UI'] == stroke_num, 'trigger_pull_amount_left'] = 0
        df.loc[df['gesture_counter_UI'] == stroke_num, 'trigger_pull_amount_right'] = 0

### Overview #4
Cumulative raw data file functions: The methods outlined below will be used to create csvs that contain cumulative raw data for each gesture. 

### Functions
* create_columns_in_cumulative_data: Initializes a df with necessary columns
* add_sub_data: Takes in a data file from a sub, session to add to the cumulative data file

In [4]:
def create_columns_in_cumulative_data(df):
    """
    Initializes an empty dataframe with columns for subject, session, trial ids as well as controller/head translation and rotations.

    Parameters
    -----
    df : dataframe
        dataframe to add new columns to
    """
    df['SubID'] = np.nan
    df['SessionID'] = np.nan
    df['TrialID'] = np.nan
    df['HeadPosX'], df['HeadPosY'], df['HeadPosZ'] = np.nan, np.nan, np.nan
    df['HeadRotX'], df['HeadRotY'], df['HeadRotZ'], df['HeadRotW'] = np.nan, np.nan, np.nan, np.nan
    df['L-HandPosX'], df['L-HandPosY'], df['L-HandPosZ'] = np.nan, np.nan, np.nan
    df['L-HandRotX'], df['L-HandRotY'], df['L-HandRotZ'], df['L-HandRotW'] = np.nan, np.nan, np.nan, np.nan
    df['R-HandPosX'], df['R-HandPosY'], df['R-HandPosZ'] = np.nan, np.nan, np.nan
    df['R-HandRotX'], df['R-HandRotY'], df['R-HandRotZ'], df['R-HandRotW'] = np.nan, np.nan, np.nan, np.nan
    df['Label'] = np.nan


def add_sub_data(main_df, file_path, gesture_index, file_index):
    """
    Adds data from sub session to the main dataframe, which is cummulative of all the data for a single gesture.
    
    Parameters
    -----
    main_df : dataframe
        dataframe that will contain all of the subject session data for a single gesture
    file_path : str
        path to a subject session data file
    gesture_index : int
        based on gesture_dict keys from overview #2. Each gesture has a corresponding index
    file_index : int
        based on gesture_dict values from overview #2. Each file has a corresponding index

    Returns
    -----
    main_df : dataframe
        the original main_df but with updated values from the file_path
    """

    base_folder_name = os.path.basename(os.path.dirname(file_path))

    # Grab subject and session id based on file name
    subID = base_folder_name.split('_')[1][3:]
    sessionID = base_folder_name.split('_')[2][4:]

    df = pd.read_csv(file_path)
    remove_strokes(df, gesture_index, file_index)

    # Rename columns for trial id, head, controller positions
    df = df.rename(columns={"gesture_counter_UI": "TrialID",
                            "head_translation_x": "HeadPosX", "head_translation_y": "HeadPosY", "head_translation_z": "HeadPosZ",
                            "head_rotation_x": "HeadRotX", "head_rotation_y": "HeadRotY", "head_rotation_z": "HeadRotZ", "head_rotation_w": "HeadRotW",
                            "l_controller_translation_x": "L-HandPosX", "l_controller_translation_y": "L-HandPosY", "l_controller_translation_z": "L-HandPosZ",
                            "l_controller_rotation_x": "L-HandRotX", "l_controller_rotation_y": "L-HandRotY", "l_controller_rotation_z": "L-HandRotZ", "l_controller_rotation_w": "L-HandRotW",
                            "r_controller_translation_x": "R-HandPosX", "r_controller_translation_y": "R-HandPosY", "r_controller_translation_z": "R-HandPosZ",
                            "r_controller_rotation_x": "R-HandRotX", "r_controller_rotation_y": "R-HandRotY", "r_controller_rotation_z": "R-HandRotZ", "r_controller_rotation_w": "R-HandRotW"})

    # Iterate through each trial to add to the main df
    for trial_num in range(5):
        # Filter rows where either trigger is pulled and matches the trial number
        filtered_df = df[(df["TrialID"] == (trial_num+1)) & 
                        ((df["trigger_pull_amount_left"] != 0) | (df["trigger_pull_amount_right"] != 0))]

        # If dataframe is empty, move to next trial
        if (len(df) == 0):
            continue

        # Adding subject and session ID to the df
        filtered_df["SubID"] = subID
        filtered_df["SessionID"] = sessionID

        # Filtering to keep important columns
        filtered_df = filtered_df.loc[:,["SubID", "SessionID", "TrialID", "HeadPosX", "HeadPosY", "HeadPosZ", "HeadRotX", "HeadRotY", "HeadRotZ", "HeadRotW", "L-HandPosX", "L-HandPosY", "L-HandPosZ", "L-HandRotX", "L-HandRotY", "L-HandRotZ", "L-HandRotW", "R-HandPosX", "R-HandPosY", "R-HandPosZ", "R-HandRotX", "R-HandRotY", "R-HandRotZ", "R-HandRotW"]]

        main_df = pd.concat([main_df, filtered_df])

    return main_df

### Overview #5
Cumulative raw data file generation: Creates the cumulative data file for each gesture. Iterates through the subject data files, identifies valid trials, and adds it to the main df. It should output 12 csvs for each of the 12 gestures.

In [365]:
# Enumerate through gesture_dict keys
for i, gesture_folder in enumerate(gesture_dict):
    # Initializing the main df to populate
    main_df = pd.DataFrame()
    create_columns_in_cumulative_data(main_df)
    
    # Enumerate through files in freeform folder only. No instructional
    for j, file_path in enumerate(gesture_dict[gesture_folder][0]):
        main_df = add_sub_data(main_df, file_path, i, j)
    
    output_path = os.path.join('..', 'PreprocessedDataML', 'RawData', gesture_folder + '.csv')
    # Save the output CSV if it doesn't already exist
    if not os.path.exists(output_path):
        main_df.to_csv(output_path, header=True, index=False)

C:\Users\katie\AppData\Local\Temp\ipykernel_44332\2276431647.py:72: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\katie\AppData\Local\Temp\ipykernel_44332\2276431647.py:73: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\katie\AppData\Local\Temp\ipykernel_44332\2276431647.py:72: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pa

### Overview #6
Label count functions: The methods outlined below will be used to create csv that contains the label count for each gesture. 

### Functions
* create_columns_in_label_count: Initializes a df with necessary columns

In [5]:
def create_columns_in_label_count(df):
    df['Gesture'] = np.nan
    df['L-Dominant'], df['L-Dominant (Other)'] = np.nan, np.nan
    df['R-Dominant'], df['R-Dominant (Other)'] = np.nan, np.nan
    df['Parallel'], df['Parallel (Other)'] = np.nan, np.nan
    df['Circular Symmetry'], df['Circular Symmetry (Other)'] = np.nan, np.nan
    df['Anti-parallel: Convergent'], df['Anti-parallel: Convergent (Other)'] = np.nan, np.nan
    df['Anti-parallel: Divergent'], df['Anti-parallel: Divergent (Other)'] = np.nan, np.nan

### Overview #7
Label count file generation: The code below takes in a folder that contains the raw cumulative data, which should've been generated in the blocks above. It looks at the first row of every primary composite key (SubID, SessionID, TrialID) per gesture and identifies its label before adding it to the label count df. It will output a csv file.

In [360]:
raw_data_folder_path = os.path.join('..', 'PreprocessedDataML', 'RawData')

label_count_df = pd.DataFrame()
create_columns_in_label_count(label_count_df)

for file_name in os.listdir(raw_data_folder_path):
    file_path = os.path.join(raw_data_folder_path, file_name)

    df = pd.read_csv(file_path)
    dict_label_count = {'Gesture': file_name.split('.')[0],
                        'L-Dominant': 0,
                        'L-Dominant (Other)': 0,
                        'R-Dominant': 0,
                        'R-Dominant (Other)': 0,
                        'Parallel': 0,
                        'Parallel (Other)': 0,
                        'Circular Symmetry': 0,
                        'Circular Symmetry (Other)': 0,
                        'Anti-parallel: Convergent': 0,
                        'Anti-parallel: Convergent (Other)': 0,
                        'Anti-parallel: Divergent': 0,
                        'Anti-parallel: Divergent (Other)': 0}

    # Group by the composite key
    grouped = df.groupby(["SubID", "SessionID", "TrialID"]).first().reset_index()

    for _, row in grouped.iterrows():
        label_val = row["Label"]
        if label_val in dict_label_count:  # Only update if d_value matches a dictionary key
            dict_label_count[label_val] += 1
        else:
            print(file_name)
            print(label_val)
            print("No key found in dictionary.")

    df_to_add = pd.DataFrame([dict_label_count])

    label_count_df = pd.concat([label_count_df, df_to_add])

output_path = os.path.join('..', 'PreprocessedDataML', 'label_count.csv')
if not os.path.exists(output_path):
    label_count_df.to_csv(output_path, header=True, index=False)


### Overview #8
Hook removal functions

In [6]:
def compute_curvature(stroke_points):
    curvatures = np.zeros(len(stroke_points))  # Initialize curvature array

    for i in range(1, len(stroke_points) - 1):
        # Compute vectors
        v1 = stroke_points[i] - stroke_points[i - 1]
        v2 = stroke_points[i + 1] - stroke_points[i]
        
        if not np.all(v1 == 0) and not np.all(v2 == 0):
            # Compute curvature as the norm of the cross product, normalized by vector magnitudes
            cross_product = np.cross(v1, v2)
            curvature = np.linalg.norm(cross_product) / (np.linalg.norm(v1) * np.linalg.norm(v2))

            # Checks if cross_product is vector 0s and updates curvature from np.nan to 0
            if np.allclose(cross_product, np.zeros(3)):
                curvature = 0

            # Add curvature for those three points to the list
            curvatures[i] = curvature
        else:
            curvatures[i] = curvatures[i-1]

    # Last point has the same curvature as the point before it
    curvatures[len(stroke_points)-1] = curvatures[len(stroke_points)-2]

    return curvatures


def remove_hooks(df, label):
    # Iterate through the first ten points in reverse and gets new start index based on sharp curvature
    start, end = 0, len(df)-1
    threshold = 2

    if label != "R-Dominant" and label != "R-Dominant (Other)":
        l_x = pd.to_numeric(df['L-HandPosX'], errors='coerce')
        l_y = pd.to_numeric(df['L-HandPosX'], errors='coerce')
        l_z = pd.to_numeric(df['L-HandPosX'], errors='coerce')

        l_stroke = np.array(list(zip(l_x,l_y,l_z)))
        l_curvatures = compute_curvature(l_stroke)

        l_median = np.median(l_curvatures)
        l_mad = np.median([abs(x - l_median) for x in l_curvatures])

        l_modified_z_scores = [0.6745 * (x - l_median) / l_mad if l_mad != 0 else 0 for x in l_curvatures]

        for i in range(int(len(l_curvatures)*2/5),-1,-1):
            if abs(l_modified_z_scores[i]) > threshold:
                start = i
                break

        # Iterate through the first last points and gets new end index based on sharp curvature
        for i in range(len(l_curvatures)-(int(len(l_curvatures)*2/5)), len(l_curvatures)):
            if abs(l_modified_z_scores[i]) > threshold:
                end = i-1
                break

    if label != "L-Dominant" and label != "L-Dominant (Other)":
        r_x = pd.to_numeric(df['R-HandPosX'], errors='coerce')
        r_y = pd.to_numeric(df['R-HandPosY'], errors='coerce')
        r_z = pd.to_numeric(df['R-HandPosZ'], errors='coerce')

        r_stroke = np.array(list(zip(r_x,r_y,r_z)))
        r_curvatures = compute_curvature(r_stroke)

        r_median = np.median(r_curvatures)
        r_mad = np.median([abs(x - r_median) for x in r_curvatures])

        r_modified_z_scores = [0.6745 * (x - r_median) / r_mad if r_mad != 0 else 0 for x in r_curvatures]

        for i in range(int(len(r_curvatures)*2/5),-1,-1):
            if (abs(r_modified_z_scores[i]) > threshold) and (i>start):
                start = i
                break
        for i in range(len(r_curvatures)-(int(len(r_curvatures)*2/5)), len(r_curvatures)):
            if (abs(r_modified_z_scores[i]) > threshold) and (i<end):
                end = i-1
                break
    
    return df.iloc[start:end]

### Overview #9
Hook removed data files generation

In [328]:
raw_data_folder = os.path.join('..', 'PreprocessedDataML', 'RawData')
output_folder = os.path.join('..', 'PreprocessedDataML', 'HookRemovedData')

for file in os.listdir(raw_data_folder):
    # Skip label count file
    if file == "LabelCount.csv":
        continue

    # Skip anything that isn't a file
    if os.path.isfile(os.path.join(raw_data_folder, file)) == False:
        continue

    # Initialize a dataframe
    df = pd.read_csv(os.path.join(raw_data_folder, file))
    new_main_df = pd.DataFrame()
    create_columns_in_cumulative_data(new_main_df)

    # Group by the sub, sess, trial id columns (composite primary key)
    grouped = df.groupby(["SubID", "SessionID", "TrialID"])

    # Iterate through each group and remove the hooks for that
    for key, group in grouped:
        label = group.iloc[0]["Label"]
        df_with_removed_hooks = remove_hooks(group, label)
        
        # Add new hooked removed df to overall df
        new_main_df = pd.concat([new_main_df, df_with_removed_hooks])

    output_path = os.path.join(output_folder, file)
    if not os.path.exists(output_path):
        new_main_df.to_csv(output_path, header=True, index=False)

### Overview #10
Egocentric functions

In [7]:
selected_columns = ['R-HandPosX', 'R-HandPosY', 'R-HandPosZ',
                    'R-HandRotX', 'R-HandRotY', 'R-HandRotZ', 'R-HandRotW',
                    'L-HandPosX', 'L-HandPosY', 'L-HandPosZ',
                    'L-HandRotX', 'L-HandRotY', 'L-HandRotZ', 'L-HandRotW',                    
                    'HeadPosX', 'HeadPosY', 'HeadPosZ',
                    'HeadRotX', 'HeadRotY', 'HeadRotZ', 'HeadRotW',]


## 3.1 L-handed -> R-handed (only invert translation_z for head + L/R hands)
def left_to_right_handed(data_sample, selected_columns):
    r_handed_data_sample = [] # r-handed data of the chosen sample; 21 lists
    for idx, col in enumerate(selected_columns):
        if 'PosZ' in col:
            inv_lst = [-float(val) for val in data_sample[idx]]
            
            r_handed_data_sample.append(inv_lst)
        else: # other columns not needing inverted
            r_handed_data_sample.append(data_sample[idx])
    return r_handed_data_sample



## 3.2 Data processing: Rotation 4-Quaternion to 3-direction for head + L/R hands
directional_data_names = ['R-HandPosX', 'R-HandPosY', 'R-HandPosZ', 'R-HandDirectionX', 'R-HandDirectionY', 'R-HandDirectionZ',
                         'L-HandPosX', 'L-HandPosY', 'L-HandPosZ', 'L-HandDirectionX', 'L-HandDirectionY', 'L-HandDirectionZ',
                         'HeadPosX', 'HeadPosY', 'HeadPosZ', 'HeadDirectionX', 'HeadDirectionY', 'HeadDirectionZ']

# function: Convert a quaternion into a 3D rotation matrix
def quaternion_rotation_matrix(Q):
    # Extract values from Q
    qx = float(Q[0])
    qy = float(Q[1])
    qz = float(Q[2])
    qw = float(Q[3])

    # First row of the rotation matrix
    r00 = 1.0 - 2.0 * (qy * qy + qz * qz)
    r01 = 2.0 * (qx * qy - qw * qz)
    r02 = 2.0 * (qx * qz + qw * qy)

    # Second row of the rotation matrix
    r10 = 2.0 * (qx * qy + qw * qz)
    r11 = 1.0 - 2.0 * (qx * qx + qz * qz)
    r12 = 2.0 * (qy * qz - qw * qx)

    # Third row of the rotation matrix
    r20 = 2.0 * (qx * qz - qw * qy)
    r21 = 2.0 * (qy * qz + qw * qx)
    r22 = 1.0 - 2.0 * (qx * qx + qy * qy)

    # 3x3 rotation matrix
    rot_matrix = np.array([[r00, r01, r02],
                           [r10, r11, r12],
                           [r20, r21, r22]])
    return rot_matrix

# function: Convert rotation data represented as quaterinons into a directions (3D vector)
#            by rotating a forward vector (0, 0, 1) using the given quaternion
def convertQuaternions2Directions(rotation_x_list, rotation_y_list,rotation_z_list,rotation_w_list):
    direction_x_list = []
    direction_y_list = []
    direction_z_list = []
    forward_vec = np.array([0, 0, 1])
    for i in range(len(rotation_x_list)):
        quaternion = [rotation_x_list[i],rotation_y_list[i],rotation_z_list[i],rotation_w_list[i]]
        rot_matrix = quaternion_rotation_matrix(quaternion)
        dir_vec = rot_matrix.dot(forward_vec)
        direction_x_list.append(dir_vec[0])
        direction_y_list.append(dir_vec[1])
        direction_z_list.append(dir_vec[2])
    return direction_x_list, direction_y_list, direction_z_list

# Apply to rotations of head and L/R
def quaternion_to_direction(r_handed_data_sample, selected_columns):
    directional_data_sample = []
    items = [[[0, 2], [3, 6]], [[7, 9], [10, 13]], [[14, 16], [17, 20]]] # index range for R/L/Head:[R:[trans,rot],L:[trans,rot],Head:[trans,rot]]
    # traverse R -> L -> Head in order
    for item in items: 
        # translation data
        for idx in range(item[0][0], item[0][1] + 1):
            directional_data_sample.append(r_handed_data_sample[idx])
        # rotation data
        quaternion_lists = []
        direction_x_list = []
        direction_y_list = []
        direction_z_list = []
        for idx in range(item[1][0], item[1][1] + 1):
            quaternion_lists.append(r_handed_data_sample[idx])
        if (len(quaternion_lists) == 4):
            direction_x_list, direction_y_list, direction_z_list = convertQuaternions2Directions(
                quaternion_lists[0], quaternion_lists[1], quaternion_lists[2], quaternion_lists[3])
            directional_data_sample.append(direction_x_list)
            directional_data_sample.append(direction_y_list)
            directional_data_sample.append(direction_z_list)
    return directional_data_sample





## 3.3 World -> Egocentric coordinates (head + L/R hands -> L/R hands) 
#   and reformat the sample as [[...],[...],...[...]], a list of 12 signals (lists)
egocentric_data_names = ['R-HandPosU', 'R-HandPosV', 'R-HandPosW', 'R-HandDirectionU', 'R-HandDirectionV', 'R-HandDirectionW',
                         'L-HandPosU', 'L-HandPosV', 'L-HandPosW', 'L-HandDirectionU', 'L-HandDirectionV', 'L-HandDirectionW']

# function construct a head space coordinate system consisting of three basis vectors, u, v, w for EACH trial
def buildHeadSpaceCoordVectors(head_direction_x_list, head_direction_y_list, head_direction_z_list):
    # average head direction vectors across all frames in the trial
    avg_head_direction_x = sum(head_direction_x_list)/len(head_direction_x_list)
    avg_head_direction_y = sum(head_direction_y_list)/len(head_direction_y_list)
    avg_head_direction_z = sum(head_direction_z_list)/len(head_direction_z_list)
    
    # calculate head space coordinate vectors, u, v, w
    head_space_w = 1.0 * np.array([avg_head_direction_x, avg_head_direction_y, avg_head_direction_z]) # w is opposite of the head direction
    head_space_v = np.array([0, 1, 0])
    head_space_u = np.cross(head_space_v, head_space_w)
    head_space_v = np.cross(head_space_w, head_space_u)

    return head_space_u, head_space_v, head_space_w

# function: convert hand data from world to head
def convert_hand_world_to_head(avg_head_translation, rot_matrix, hand_translation_x_list,  hand_translation_y_list,  hand_translation_z_list,
                                hand_direction_x_list,  hand_direction_y_list,  hand_direction_z_list):
    hand_translation_u_list = []
    hand_translation_v_list = []
    hand_translation_w_list = []
    hand_direction_u_list = []
    hand_direction_v_list = []
    hand_direction_w_list = []

    # convert hand data world -> space, frame by frame
    for idx in range(len(hand_translation_x_list)):
        # Translation data: (Already correct)
        hand_translation_u = float(hand_translation_x_list[idx]) - float(avg_head_translation[0])
        hand_translation_v = float(hand_translation_y_list[idx]) - float(avg_head_translation[1])
        hand_translation_w = float(hand_translation_z_list[idx]) - float(avg_head_translation[2])
        hand_translation = rot_matrix.dot(np.array([hand_translation_u, hand_translation_v, hand_translation_w]))
        hand_translation_u_list.append(hand_translation[0])
        hand_translation_v_list.append(hand_translation[1])
        hand_translation_w_list.append(hand_translation[2])

        # Direction data: (Change this to point toward the origin)
        # Compute the direction from the hand position to the origin
        direction_to_head = 1.0 * np.array([hand_translation_u, hand_translation_v, hand_translation_w])  # Direction to the origin
        hand_direction_u_list.append(direction_to_head[0])
        hand_direction_v_list.append(direction_to_head[1])
        hand_direction_w_list.append(direction_to_head[2])

    return hand_translation_u_list, hand_translation_v_list, hand_translation_w_list, hand_direction_u_list, hand_direction_v_list, hand_direction_w_list

def world_to_headspace(directional_data_sample, directional_data_names):
    egocentric_data_sample = []
    # obtain head translation data in world space
    # converts to float bc for some reason, some of the data is are strings
    head_translation_x_list = [float(i) for i in directional_data_sample[12]]
    head_translation_y_list = [float(i) for i in directional_data_sample[13]]
    head_translation_z_list = [float(i) for i in directional_data_sample[14]]

    # average head translation data -> averaged head position in world
    avg_head_translation = []
    avg_head_translation.append(sum(head_translation_x_list)/len(head_translation_x_list))
    avg_head_translation.append(sum(head_translation_y_list)/len(head_translation_y_list))
    avg_head_translation.append(sum(head_translation_z_list)/len(head_translation_z_list))

    # obtain head direction data in world space
    head_direction_x_list = directional_data_sample[15]
    head_direction_y_list = directional_data_sample[16]
    head_direction_z_list = directional_data_sample[17]
    
    # build coordiante vectors of the head space: u, v, w
    head_space_u, head_space_v, head_space_w = buildHeadSpaceCoordVectors(head_direction_x_list=head_direction_x_list, 
                                                                          head_direction_y_list=head_direction_y_list,
                                                                          head_direction_z_list=head_direction_z_list)

    # construct rotation matrix transforming vectors from world to head space
    rot_matrix = np.array([head_space_u.tolist(), # u
                           head_space_v.tolist(), # v
                           head_space_w.tolist()]) # w
    
    # convert RIGHT hand data from world to head space
    r_translation_u_list, r_translation_v_list, r_translation_w_list, \
       r_direction_u_list,r_direction_v_list, r_direction_w_list \
            = convert_hand_world_to_head(avg_head_translation, rot_matrix, directional_data_sample[0], directional_data_sample[1],directional_data_sample[2],\
                                         directional_data_sample[3], directional_data_sample[4],directional_data_sample[5])

    # convert LEFT hand data from world to head space
    l_translation_u_list, l_translation_v_list, l_translation_w_list, \
        l_direction_u_list, l_direction_v_list, l_direction_w_list \
            = convert_hand_world_to_head(avg_head_translation, rot_matrix, directional_data_sample[6], directional_data_sample[7],directional_data_sample[8],\
                                         directional_data_sample[9], directional_data_sample[10],directional_data_sample[11])
    
    # add the converted data
    egocentric_data_sample.append(r_translation_u_list) # right hand
    egocentric_data_sample.append(r_translation_v_list)
    egocentric_data_sample.append(r_translation_w_list)
    egocentric_data_sample.append(r_direction_u_list)
    egocentric_data_sample.append(r_direction_v_list)
    egocentric_data_sample.append(r_direction_w_list)
    egocentric_data_sample.append(l_translation_u_list) # left hand
    egocentric_data_sample.append(l_translation_v_list)
    egocentric_data_sample.append(l_translation_w_list)
    egocentric_data_sample.append(l_direction_u_list)
    egocentric_data_sample.append(l_direction_v_list)
    egocentric_data_sample.append(l_direction_w_list)

    return egocentric_data_sample

### Overview #11
Hook removed egocentralized data functions:

In [8]:
def create_columns_in_hook_removed_egocentralized_data(df):
    """
    Initializes an empty dataframe with columns for subject, session, trial ids as well as controller/head translation and rotations.

    Parameters
    -----
    df : dataframe
        dataframe to add new columns to
    """
    df['SubID'] = np.nan
    df['SessionID'] = np.nan
    df['TrialID'] = np.nan
    df['HeadPosX'], df['HeadPosY'], df['HeadPosZ'] = np.nan, np.nan, np.nan
    df['HeadRotX'], df['HeadRotY'], df['HeadRotZ'], df['HeadRotW'] = np.nan, np.nan, np.nan, np.nan
    df['L-HandPosU'], df['L-HandPosV'], df['L-HandPosW'] = np.nan, np.nan, np.nan
    df['L-HandDirectionU'], df['L-HandDirectionV'], df['L-HandDirectionW']= np.nan, np.nan, np.nan
    df['R-HandPosU'], df['R-HandPosV'], df['R-HandPosW'] = np.nan, np.nan, np.nan
    df['R-HandDirectionU'], df['R-HandDirectionV'], df['R-HandDirectionW'] = np.nan, np.nan, np.nan
    df['Label'] = np.nan

### Overview #12
Hook removed egocentralized data files generation. Takes hooked removed data folder and iterates through each group filtered by sub, sess, trial to egocentralize.

In [ ]:
hook_removed_data_folder = os.path.join('..', 'PreprocessedDataML', 'HookRemovedData')
output_folder = os.path.join('..', 'PreprocessedDataML', 'HookRemovedEgocentralizedData')

for file in os.listdir(hook_removed_data_folder):
    if os.path.isfile(os.path.join(hook_removed_data_folder, file)) == False:
        continue
    df = pd.read_csv(os.path.join(hook_removed_data_folder, file))
    new_main_df = pd.DataFrame()
    create_columns_in_hook_removed_egocentralized_data(new_main_df)

    # Group by the sub, sess, trial id columns (composite primary key)
    grouped = df.groupby(["SubID", "SessionID", "TrialID"])

    # Iterate through each group from hook removed data, getting the egocentralized positions and directions, and adding to cumulative data
    for key, group in grouped:
        # Egocentralizing the data
        data_sample = [] # original data of the chosen sample; 21 lists
        for col in selected_columns:
            data_sample.append(group[col].tolist())
        
        # 3.2
        directional_data_sample = quaternion_to_direction(r_handed_data_sample=data_sample, selected_columns=selected_columns)
        # 3.3
        egocentric_data_sample = world_to_headspace(directional_data_sample=directional_data_sample, directional_data_names=directional_data_names)
        
        # For reference
        #left_points = (list(zip(egocentric_data_sample[6], egocentric_data_sample[7], egocentric_data_sample[8])))
        #right_points = (list(zip(egocentric_data_sample[0], egocentric_data_sample[1], egocentric_data_sample[2])))
        
        # Filtering the group to rename columns from rotation to direction for egocentric output since there are excessive columns in rotation
        group = group.loc[:,["SubID", "SessionID", "TrialID", "HeadPosX", "HeadPosY", "HeadPosZ", "HeadRotX", "HeadRotY", "HeadRotZ", "HeadRotW", "L-HandPosX", "L-HandPosY", "L-HandPosZ", "L-HandRotX", "L-HandRotY", "L-HandRotZ", "R-HandPosX", "R-HandPosY", "R-HandPosZ", "R-HandRotX", "R-HandRotY", "R-HandRotZ", "Label"]]

        group = group.rename(columns={
                            "L-HandPosX": "L-HandPosU", "L-HandPosY": "L-HandPosV", "L-HandPosZ": "L-HandPosW",
                            "L-HandRotX": "L-HandDirectionU", "L-HandRotY": "L-HandDirectionV", "L-HandRotZ": "L-HandDirectionW",
                            "R-HandPosX": "R-HandPosU", "R-HandPosY": "R-HandPosV", "R-HandPosZ": "R-HandPosW",
                            "R-HandRotX": "R-HandDirectionU", "R-HandRotY": "R-HandDirectionV", "R-HandRotZ": "R-HandDirectionW"})
        
        # Update values for egocentric position & directions
        group['L-HandPosU'], group['L-HandPosV'], group['L-HandPosW'] = egocentric_data_sample[6], egocentric_data_sample[7], egocentric_data_sample[8]
        group['L-HandDirectionU'], group['L-HandDirectionV'], group['L-HandDirectionW'] = egocentric_data_sample[9], egocentric_data_sample[10], egocentric_data_sample[11]
        group['R-HandPosU'], group['R-HandPosV'], group['R-HandPosW'] = egocentric_data_sample[0], egocentric_data_sample[1], egocentric_data_sample[2]
        group['R-HandDirectionU'], group['R-HandDirectionV'], group['R-HandDirectionW'] = egocentric_data_sample[3], egocentric_data_sample[4], egocentric_data_sample[5]

        # Add new egocentralized hooked removed df to overall df
        new_main_df = pd.concat([new_main_df, group])

    output_path = os.path.join(output_folder, file)
    if not os.path.exists(output_path):
        new_main_df.to_csv(output_path, header=True, index=False)

### Overview #13
Summary statistic extraction functions

In [9]:
def create_columns_in_metric_summary(df):
    df['SubID'] = np.nan
    df['SessionID'] = np.nan
    df['TrialID'] = np.nan
    df['L-Min'] = np.nan
    df['L-Max'] = np.nan
    df['L-Mean'] = np.nan
    df['L-Std'] = np.nan
    df['R-Min'] = np.nan
    df['R-Max'] = np.nan
    df['R-Mean'] = np.nan
    df['R-Std'] = np.nan

def create_columns_in_metric_summary_xyz(df):
    df['SubID'] = np.nan
    df['SessionID'] = np.nan
    df['TrialID'] = np.nan
    df['L-U_Min'] = np.nan
    df['L-V_Min'] = np.nan
    df['L-W_Min'] = np.nan
    df['L-U_Max'] = np.nan
    df['L-V_Max'] = np.nan
    df['L-W_Max'] = np.nan
    df['L-U_Mean'] = np.nan
    df['L-V_Mean'] = np.nan
    df['L-W_Mean'] = np.nan
    df['L-U_Std'] = np.nan
    df['L-V_Std'] = np.nan
    df['L-W_Std'] = np.nan
    df['R-U_Min'] = np.nan
    df['R-V_Min'] = np.nan
    df['R-W_Min'] = np.nan
    df['R-U_Max'] = np.nan
    df['R-V_Max'] = np.nan
    df['R-W_Max'] = np.nan
    df['R-U_Mean'] = np.nan
    df['R-V_Mean'] = np.nan
    df['R-W_Mean'] = np.nan
    df['R-U_Std'] = np.nan
    df['R-V_Std'] = np.nan
    df['R-W_Std'] = np.nan

# Takes main dataframe, looks at the current sub session file, and adds data from that to the main df
def get_raw_data(main_df, file_path, gesture_index, file_index):
    base_folder_name = os.path.basename(os.path.dirname(file_path))

    # Grab subject and session id based on file name
    subID = base_folder_name.split('_')[1][3:]
    sessionID = base_folder_name.split('_')[2][4:]

    df = pd.read_csv(file_path)
    remove_strokes(df, gesture_index, file_index)

    # Rename columns for trial id, head, controller positions
    df = df.rename(columns={"time": "Time",
                            "gesture_counter_UI": "TrialID",
                            "head_translation_x": "HeadPosX", "head_translation_y": "HeadPosY", "head_translation_z": "HeadPosZ",
                            "head_rotation_x": "HeadRotX", "head_rotation_y": "HeadRotY", "head_rotation_z": "HeadRotZ", "head_rotation_w": "HeadRotW",
                            "l_controller_translation_x": "L-HandPosX", "l_controller_translation_y": "L-HandPosY", "l_controller_translation_z": "L-HandPosZ",
                            "l_controller_rotation_x": "L-HandRotX", "l_controller_rotation_y": "L-HandRotY", "l_controller_rotation_z": "L-HandRotZ", "l_controller_rotation_w": "L-HandRotW",
                            "r_controller_translation_x": "R-HandPosX", "r_controller_translation_y": "R-HandPosY", "r_controller_translation_z": "R-HandPosZ",
                            "r_controller_rotation_x": "R-HandRotX", "r_controller_rotation_y": "R-HandRotY", "r_controller_rotation_z": "R-HandRotZ", "r_controller_rotation_w": "R-HandRotW"})

    # Iterate through each trial to add to the main df
    for trial_num in range(5):
        # Filter rows where either trigger is pulled and matches the trial number
        filtered_df = df[(df["TrialID"] == (trial_num+1)) & 
                        ((df["trigger_pull_amount_left"] != 0) | (df["trigger_pull_amount_right"] != 0))]

        # If dataframe is empty, move to next trial
        if (len(df) == 0):
            continue

        # Adding subject and session ID to the df
        filtered_df["SubID"] = int(subID)
        filtered_df["SessionID"] = int(sessionID)

        # Filtering to keep important columns
        filtered_df = filtered_df.loc[:,["Time", "SubID", "SessionID", "TrialID", "HeadPosX", "HeadPosY", "HeadPosZ", "HeadRotX", "HeadRotY", "HeadRotZ", "HeadRotW", "L-HandPosX", "L-HandPosY", "L-HandPosZ", "L-HandRotX", "L-HandRotY", "L-HandRotZ", "L-HandRotW", "R-HandPosX", "R-HandPosY", "R-HandPosZ", "R-HandRotX", "R-HandRotY", "R-HandRotZ", "R-HandRotW"]]
        filtered_df["TrialID"] = filtered_df["TrialID"].astype(int)
        
        main_df = pd.concat([main_df, filtered_df])

    return main_df

def add_label_to_data(labeled_df, unlabeled_df):
    labeled_df_grouped = labeled_df.groupby(["SubID", "SessionID", "TrialID"]).first().reset_index()
    unlabeled_df_grouped = unlabeled_df.groupby(["SubID", "SessionID", "TrialID"])

    for (sub_id, sess_id, trial_id), group_df in unlabeled_df_grouped:
        label = labeled_df_grouped.loc[(labeled_df_grouped["SubID"] == int(sub_id)) & (labeled_df_grouped["SessionID"] == int(sess_id)) & (labeled_df_grouped["TrialID"] == int(trial_id)), "Label"].iloc[0]      
        unlabeled_df.loc[group_df.index, "Label"] = label

    return unlabeled_df

def get_hook_removed_data(labeled_df, raw_data_df):
    new_df = pd.DataFrame()

    # Add labels prior to hook removal
    raw_data_df = add_label_to_data(labeled_df, raw_data_df)
    
    # Iterating through raw data groups to remove hooks and add label
    for key, group in raw_data_df.groupby(["SubID", "SessionID", "TrialID"]):
        df_with_removed_hooks = remove_hooks(group, group.iloc[0]['Label'])

        # Add new hooked removed df to overall df
        new_df = pd.concat([new_df, df_with_removed_hooks])

    return new_df

def get_hook_removed_egocentralized_data(hook_removed_df):
    new_df = pd.DataFrame()

    # Group by the sub, sess, trial id columns (composite primary key)
    grouped = hook_removed_df.groupby(["SubID", "SessionID", "TrialID"])

    # Iterate through each group from hook removed data, getting the egocentralized positions and directions, and adding to cumulative data
    for key, group in grouped:
        # Egocentralizing the data
        data_sample = [] # original data of the chosen sample; 21 lists
        for col in selected_columns:
            data_sample.append(group[col].tolist())
        
        # 3.2
        directional_data_sample = quaternion_to_direction(r_handed_data_sample=data_sample, selected_columns=selected_columns)
        # 3.3
        egocentric_data_sample = world_to_headspace(directional_data_sample=directional_data_sample, directional_data_names=directional_data_names)
        
        # For reference
        #left_points = (list(zip(egocentric_data_sample[6], egocentric_data_sample[7], egocentric_data_sample[8])))
        #right_points = (list(zip(egocentric_data_sample[0], egocentric_data_sample[1], egocentric_data_sample[2])))
        
        # Filtering the group to rename columns from rotation to direction for egocentric output since there are excessive columns in rotation
        group = group.loc[:,["SubID", "SessionID", "TrialID", "Time", "HeadPosX", "HeadPosY", "HeadPosZ", "HeadRotX", "HeadRotY", "HeadRotZ", "HeadRotW", "L-HandPosX", "L-HandPosY", "L-HandPosZ", "L-HandRotX", "L-HandRotY", "L-HandRotZ", "R-HandPosX", "R-HandPosY", "R-HandPosZ", "R-HandRotX", "R-HandRotY", "R-HandRotZ", "Label"]]

        group = group.rename(columns={
                            "L-HandPosX": "L-HandPosU", "L-HandPosY": "L-HandPosV", "L-HandPosZ": "L-HandPosW",
                            "L-HandRotX": "L-HandDirectionU", "L-HandRotY": "L-HandDirectionV", "L-HandRotZ": "L-HandDirectionW",
                            "R-HandPosX": "R-HandPosU", "R-HandPosY": "R-HandPosV", "R-HandPosZ": "R-HandPosW",
                            "R-HandRotX": "R-HandDirectionU", "R-HandRotY": "R-HandDirectionV", "R-HandRotZ": "R-HandDirectionW"})
        
        # Update values for egocentric position & directions
        group['L-HandPosU'], group['L-HandPosV'], group['L-HandPosW'] = egocentric_data_sample[6], egocentric_data_sample[7], egocentric_data_sample[8]
        group['L-HandDirectionU'], group['L-HandDirectionV'], group['L-HandDirectionW'] = egocentric_data_sample[9], egocentric_data_sample[10], egocentric_data_sample[11]
        group['R-HandPosU'], group['R-HandPosV'], group['R-HandPosW'] = egocentric_data_sample[0], egocentric_data_sample[1], egocentric_data_sample[2]
        group['R-HandDirectionU'], group['R-HandDirectionV'], group['R-HandDirectionW'] = egocentric_data_sample[3], egocentric_data_sample[4], egocentric_data_sample[5]

        # Add new egocentralized hooked removed df to overall df
        new_df = pd.concat([new_df, group])
    
    return new_df

In [12]:
def get_duration_statistics(data_df, gesture):
    # Create summary statistics df
    duration_df = pd.DataFrame()
    duration_df["SubID"] = np.nan
    duration_df["SessionID"] = np.nan
    duration_df["TrialID"] = np.nan
    duration_df["Time"] = np.nan

    # Group by the sub, sess, trial id columns (composite primary key)
    grouped = data_df.groupby(["SubID", "SessionID", "TrialID"])

    # Iterate through each group
    for (sub_id, sess_id, trial_id), group in grouped:
        final_time = group.iloc[-1]["Time"]
        start_time = group.iloc[0]["Time"]

        new_row = pd.DataFrame({'SubID': [sub_id], 'SessionID': [sess_id], 'TrialID': [trial_id], 'Label': group.iloc[-1]['Label'],  "Time": [final_time-start_time]})
        
        # Append data from the group to overall df
        duration_df = pd.concat([duration_df, new_row])

    duration_df['Gesture'] = gesture

    return duration_df

def get_length_statistics(data_df, gesture):
    # Create summary statistics df
    length_df = pd.DataFrame()
    create_columns_in_metric_summary(length_df)

    # Group by the sub, sess, trial id columns (composite primary key)
    grouped = data_df.groupby(["SubID", "SessionID", "TrialID"])

    # Iterate through each stroke
    for (sub_id, sess_id, trial_id), group in grouped:
        # Compute the Euclidean distance between two rows
        l_distances = np.linalg.norm(group[['L-HandPosU', 'L-HandPosV', 'L-HandPosW']].diff().iloc[1:], axis=1)
        r_distances = np.linalg.norm(group[['R-HandPosU', 'R-HandPosV', 'R-HandPosW']].diff().iloc[1:], axis=1)

        # New data to add to metric summary
        new_row = {'SubID': int(sub_id), 'SessionID': int(sess_id), 'TrialID': int(trial_id), 'Label': group.iloc[-1]['Label'],
                   'L-Min': np.min(l_distances), 'L-Max': np.max(l_distances), 
                   'L-Mean': np.mean(l_distances), 'L-Std': np.std(l_distances),
                   'R-Min': np.min(r_distances), 'R-Max': np.max(r_distances), 
                   'R-Mean': np.mean(r_distances), 'R-Std': np.std(r_distances)}
        
        length_df = pd.concat([length_df, pd.DataFrame([new_row])])

    length_df['Gesture'] = gesture

    return length_df

def get_speed_statistics(data_df, gesture):
    speed_df = pd.DataFrame()
    create_columns_in_metric_summary(speed_df)

    # Group by the sub, sess, trial id columns (composite primary key)
    grouped = data_df.groupby(["SubID", "SessionID", "TrialID"])

    # Iterate through each stroke
    for (sub_id, sess_id, trial_id), group in grouped:
        # Compute the Euclidean distance between two rows
        l_distances = np.linalg.norm(group[['L-HandPosU', 'L-HandPosV', 'L-HandPosW']].diff().iloc[1:], axis=1)
        r_distances = np.linalg.norm(group[['R-HandPosU', 'R-HandPosV', 'R-HandPosW']].diff().iloc[1:], axis=1)
        
        # Compute time difference between two rows
        time_differences = group['Time'].diff().iloc[1:]

        # Compute speed
        l_speeds = l_distances / time_differences.values
        r_speeds = r_distances / time_differences.values

        # New data to add to metric summary
        new_row = {'SubID': int(sub_id), 'SessionID': int(sess_id), 'TrialID': int(trial_id), 'Label': group.iloc[-1]['Label'],
                   'L-Min': np.min(l_speeds), 'L-Max': np.max(l_speeds), 
                   'L-Mean': np.mean(l_speeds), 'L-Std': np.std(l_speeds),
                   'R-Min': np.min(r_speeds), 'R-Max': np.max(r_speeds), 
                   'R-Mean': np.mean(r_speeds), 'R-Std': np.std(r_speeds)}
        
        speed_df = pd.concat([speed_df, pd.DataFrame([new_row])])
    
    speed_df['Gesture'] = gesture

    return speed_df
        
def get_velocity_statistics(data_df, gesture):
    velocity_df = pd.DataFrame()
    create_columns_in_metric_summary_xyz(velocity_df)

    # Group by the sub, sess, trial id columns (composite primary key)
    grouped = data_df.groupby(["SubID", "SessionID", "TrialID"])

    # Iterate through each stroke
    for (sub_id, sess_id, trial_id), group in grouped:
        # Compute position differences between two rows
        l_u_distances = group['L-HandPosU'].diff().iloc[1:].to_numpy()
        l_v_distances = group['L-HandPosV'].diff().iloc[1:].to_numpy()
        l_w_distances = group['L-HandPosW'].diff().iloc[1:].to_numpy()
        r_u_distances = group['R-HandPosU'].diff().iloc[1:].to_numpy()
        r_v_distances = group['R-HandPosV'].diff().iloc[1:].to_numpy()
        r_w_distances = group['R-HandPosW'].diff().iloc[1:].to_numpy()

        # Compute time difference between two rows
        time_differences = group['Time'].diff().iloc[1:].to_numpy().reshape(-1, 1).flatten()

        # Compute velocity
        l_u_velocities = l_u_distances / time_differences
        l_v_velocities = l_v_distances / time_differences
        l_w_velocities = l_w_distances / time_differences
        r_u_velocities = r_u_distances / time_differences
        r_v_velocities = r_v_distances / time_differences
        r_w_velocities = r_w_distances / time_differences

        # New data to add to metric summary
        new_row = {'SubID': int(sub_id), 'SessionID': int(sess_id), 'TrialID': int(trial_id), 'Label': group.iloc[-1]['Label'],
                   'L-U_Min': np.min(l_u_velocities), 'L-V_Min': np.min(l_v_velocities), 'L-W_Min': np.min(l_w_velocities),
                   'L-U_Max': np.max(l_u_velocities), 'L-V_Max': np.max(l_v_velocities), 'L-W_Max': np.max(l_w_velocities),
                   'L-U_Mean': np.mean(l_u_velocities), 'L-V_Mean': np.mean(l_v_velocities), 'L-W_Mean': np.mean(l_w_velocities),
                   'L-U_Std': np.std(l_u_velocities), 'L-V_Std': np.std(l_v_velocities), 'L-W_Std': np.std(l_w_velocities),
                   'R-U_Min': np.min(r_u_velocities), 'R-V_Min': np.min(r_v_velocities), 'R-W_Min': np.min(r_w_velocities),
                   'R-U_Max': np.max(r_u_velocities), 'R-V_Max': np.max(r_v_velocities), 'R-W_Max': np.max(r_w_velocities),
                   'R-U_Mean': np.mean(r_u_velocities), 'R-V_Mean': np.mean(r_v_velocities), 'R-W_Mean': np.mean(r_w_velocities),
                   'R-U_Std': np.std(r_u_velocities), 'R-V_Std': np.std(r_v_velocities), 'R-W_Std': np.std(r_w_velocities)
                   }
        
        velocity_df = pd.concat([velocity_df, pd.DataFrame([new_row])])
    
    velocity_df['Gesture'] = gesture

    return velocity_df

def get_acceleration_statistics(data_df, gesture):
    acceleration_df = pd.DataFrame()
    create_columns_in_metric_summary_xyz(acceleration_df)

    # Group by the sub, sess, trial id columns (composite primary key)
    grouped = data_df.groupby(["SubID", "SessionID", "TrialID"])

    # Iterate through each stroke
    for (sub_id, sess_id, trial_id), group in grouped:
       # Compute position differences between two rows
        l_u_distances = group['L-HandPosU'].diff().iloc[1:].to_numpy()
        l_v_distances = group['L-HandPosV'].diff().iloc[1:].to_numpy()
        l_w_distances = group['L-HandPosW'].diff().iloc[1:].to_numpy()
        r_u_distances = group['R-HandPosU'].diff().iloc[1:].to_numpy()
        r_v_distances = group['R-HandPosV'].diff().iloc[1:].to_numpy()
        r_w_distances = group['R-HandPosW'].diff().iloc[1:].to_numpy()

        # Compute time difference between two rows 
        time_differences = group['Time'].diff().iloc[1:].to_numpy().reshape(-1, 1).flatten()

        # Compute acceleration
        l_u_accelerations = l_u_distances / (time_differences**2)
        l_v_accelerations = l_v_distances / (time_differences**2)
        l_w_accelerations = l_w_distances / (time_differences**2)
        r_u_accelerations = r_u_distances / (time_differences**2)
        r_v_accelerations = r_v_distances / (time_differences**2)
        r_w_accelerations = r_w_distances / (time_differences**2)

        # New data to add to metric summary
        new_row = {'SubID': int(sub_id), 'SessionID': int(sess_id), 'TrialID': int(trial_id), 'Label': group.iloc[-1]['Label'],
                   'L-U_Min': np.min(l_u_accelerations), 'L-V_Min': np.min(l_v_accelerations), 'L-W_Min': np.min(l_w_accelerations),
                   'L-U_Max': np.max(l_u_accelerations), 'L-V_Max': np.max(l_v_accelerations), 'L-W_Max': np.max(l_w_accelerations),
                   'L-U_Mean': np.mean(l_u_accelerations), 'L-V_Mean': np.mean(l_v_accelerations), 'L-W_Mean': np.mean(l_w_accelerations),
                   'L-U_Std': np.std(l_u_accelerations), 'L-V_Std': np.std(l_v_accelerations), 'L-W_Std': np.std(l_w_accelerations),
                   'R-U_Min': np.min(r_u_accelerations), 'R-V_Min': np.min(r_v_accelerations), 'R-W_Min': np.min(r_w_accelerations),
                   'R-U_Max': np.max(r_u_accelerations), 'R-V_Max': np.max(r_v_accelerations), 'R-W_Max': np.max(r_w_accelerations),
                   'R-U_Mean': np.mean(r_u_accelerations), 'R-V_Mean': np.mean(r_v_accelerations), 'R-W_Mean': np.mean(r_w_accelerations),
                   'R-U_Std': np.std(r_u_accelerations), 'R-V_Std': np.std(r_v_accelerations), 'R-W_Std': np.std(r_w_accelerations)
                   }
        
        acceleration_df = pd.concat([acceleration_df, pd.DataFrame([new_row])])
    
    acceleration_df['Gesture'] = gesture

    return acceleration_df
    
def compute_angles(stroke_points):
    angles = np.zeros(len(stroke_points))

    for i in range(1, len(stroke_points) - 1):
        # Convert points to numpy arrays for easier vector operations
        p1, p2, p3 = stroke_points[i-1], stroke_points[i], stroke_points[i+1]

        # Convert two points into a vector
        a_norm = np.linalg.norm(p2 - p1)
        b_norm = np.linalg.norm(p3 - p2)
        
        # Only add valid angles. Some errors may be causes from having two of the same points
        if (a_norm > 0.0) and (b_norm > 0.0):
            # Normalize vector a and b
            normalized_a = (p2-p1)/a_norm
            normalized_b = (p3-p2)/b_norm
            normalized_dot = np.dot(normalized_a, normalized_b)

            # Clamp dot product for floating-point safety
            normalized_dot = np.clip(normalized_dot, -1.0, 1.0)

            # Compute angle in degrees
            ext_angle = np.rad2deg(np.arccos(normalized_dot))

            angles[i] = ext_angle
            
    # Ending point has the same angle as the previous
    angles[len(stroke_points)-1] = angles[len(stroke_points)-2]

    return angles
    
def get_angle_statistics(df, gesture):
    angle_df = pd.DataFrame()
    create_columns_in_metric_summary(angle_df)

    # Group by the sub, sess, trial id columns (composite primary key)
    grouped = df.groupby(["SubID", "SessionID", "TrialID"])

    # Iterate through each group
    for (sub_id, sess_id, trial_id), group in grouped:
        # Grabbing x, y, z positions from right and left controllers
        l_x = pd.to_numeric(group['L-HandPosU'], errors='coerce')
        l_y = pd.to_numeric(group['L-HandPosV'], errors='coerce')
        l_z = pd.to_numeric(group['L-HandPosW'], errors='coerce')
        r_x = pd.to_numeric(group['R-HandPosU'], errors='coerce')
        r_y = pd.to_numeric(group['R-HandPosV'], errors='coerce')
        r_z = pd.to_numeric(group['R-HandPosW'], errors='coerce')

        # Converting independent x, y, z columns into a list of tuples
        l_stroke = np.array(list(zip(l_x,l_y,l_z)))
        r_stroke = np.array(list(zip(r_x,r_y,r_z)))

        l_angles = compute_angles(l_stroke)
        r_angles = compute_angles(r_stroke)
        
        new_row = pd.DataFrame({'SubID': [sub_id], 'SessionID': [sess_id], 'TrialID': [trial_id], 'Label': group.iloc[-1]['Label'], 
                                'L-Min': np.min(l_angles), 'L-Max': np.max(l_angles), 'L-Mean': np.mean(l_angles), 'L-Std': np.std(l_angles),
                                'R-Min': np.min(r_angles), 'R-Max': np.max(r_angles), 'R-Mean': np.mean(r_angles), 'R-Std': np.std(r_angles)})
        
        # Append data from the group to overall df
        angle_df = pd.concat([angle_df, new_row])

    angle_df['Gesture'] = gesture

    return angle_df

def get_curvature_statistics(df, gesture):
    curvature_df = pd.DataFrame()
    create_columns_in_metric_summary(curvature_df)

    # Group by the sub, sess, trial id columns (composite primary key)
    grouped = df.groupby(["SubID", "SessionID", "TrialID"])

    # Iterate through each group
    for (sub_id, sess_id, trial_id), group in grouped:
        # Grabbing x, y, z positions from right and left controllers
        l_x = pd.to_numeric(group['L-HandPosU'], errors='coerce')
        l_y = pd.to_numeric(group['L-HandPosV'], errors='coerce')
        l_z = pd.to_numeric(group['L-HandPosW'], errors='coerce')
        r_x = pd.to_numeric(group['R-HandPosU'], errors='coerce')
        r_y = pd.to_numeric(group['R-HandPosV'], errors='coerce')
        r_z = pd.to_numeric(group['R-HandPosW'], errors='coerce')

        # Converting independent x, y, z columns into a list of tuples
        l_stroke = np.array(list(zip(l_x,l_y,l_z)))
        r_stroke = np.array(list(zip(r_x,r_y,r_z)))

        # Compute curvature
        l_curvatures = compute_curvature(l_stroke)
        r_curvatures = compute_curvature(r_stroke)

        # New data to add to metric summary
        new_row = {'SubID': int(sub_id), 'SessionID': int(sess_id), 'TrialID': int(trial_id), 'Label': group.iloc[-1]['Label'],
                   'L-Min': np.min(l_curvatures), 'L-Max': np.max(l_curvatures), 
                   'L-Mean': np.mean(l_curvatures), 'L-Std': np.std(l_curvatures),
                   'R-Min': np.min(r_curvatures), 'R-Max': np.max(r_curvatures), 
                   'R-Mean': np.mean(r_curvatures), 'R-Std': np.std(r_curvatures)}
        
        curvature_df = pd.concat([curvature_df, pd.DataFrame([new_row])])

    curvature_df['Gesture'] = gesture

    return curvature_df

Only for duration bc time column is necessary

In [14]:
def create_columns_in_cumulative_data_2(df):
    """
    Initializes an empty dataframe with columns for subject, session, trial ids as well as controller/head translation and rotations.

    Parameters
    -----
    df : dataframe
        dataframe to add new columns to
    """
    df['SubID'] = np.nan
    df['SessionID'] = np.nan
    df['TrialID'] = np.nan
    df['Time'] = np.nan
    df['HeadPosX'], df['HeadPosY'], df['HeadPosZ'] = np.nan, np.nan, np.nan
    df['HeadRotX'], df['HeadRotY'], df['HeadRotZ'], df['HeadRotW'] = np.nan, np.nan, np.nan, np.nan
    df['L-HandPosX'], df['L-HandPosY'], df['L-HandPosZ'] = np.nan, np.nan, np.nan
    df['L-HandRotX'], df['L-HandRotY'], df['L-HandRotZ'], df['L-HandRotW'] = np.nan, np.nan, np.nan, np.nan
    df['R-HandPosX'], df['R-HandPosY'], df['R-HandPosZ'] = np.nan, np.nan, np.nan
    df['R-HandRotX'], df['R-HandRotY'], df['R-HandRotZ'], df['R-HandRotW'] = np.nan, np.nan, np.nan, np.nan
    df['Label'] = np.nan

labeled_data_folder = os.path.join('..', 'PreprocessedDataML', 'RawData')
labeled_files = sorted([f for f in os.listdir(labeled_data_folder) if os.path.isfile(os.path.join(labeled_data_folder, f))])

duration_df = pd.DataFrame()
length_df = pd.DataFrame()
speed_df = pd.DataFrame()
velocity_df = pd.DataFrame()
acceleration_df = pd.DataFrame()
angle_df = pd.DataFrame()
curvature_df = pd.DataFrame()

# Enumerate through each gesture
for i, gesture in enumerate(gesture_dict):
    # Initializing the main df to populate
    raw_data_df = pd.DataFrame()
    create_columns_in_cumulative_data_2(raw_data_df)

    # Enumerate through files in freeform folder only. No instructional
    for j, file_path in enumerate(gesture_dict[gesture][0]):
        raw_data_df = get_raw_data(raw_data_df, file_path, i, j)

    # Get hooked removed data with raw and labeled data
    labeled_df = pd.read_csv(os.path.join(labeled_data_folder, labeled_files[i]))
    hook_removed_df = get_hook_removed_data(labeled_df, raw_data_df)
    hook_removed_egocentralized_df = get_hook_removed_egocentralized_data(hook_removed_df)

    duration_df = pd.concat([duration_df, get_duration_statistics(hook_removed_egocentralized_df, gesture)])
    length_df = pd.concat([length_df, get_length_statistics(hook_removed_egocentralized_df, gesture)])
    speed_df = pd.concat([speed_df, get_speed_statistics(hook_removed_egocentralized_df, gesture)])
    velocity_df = pd.concat([velocity_df, get_velocity_statistics(hook_removed_egocentralized_df, gesture)])
    acceleration_df = pd.concat([acceleration_df, get_acceleration_statistics(hook_removed_egocentralized_df, gesture)])
    angle_df = pd.concat([angle_df, get_angle_statistics(hook_removed_egocentralized_df, gesture)])
    curvature_df = pd.concat([curvature_df, get_curvature_statistics(hook_removed_egocentralized_df, gesture)])

output_path = os.path.join('..', 'PreprocessedDataML', 'SummaryStatistics')
if not os.path.exists(os.path.join(output_path, 'DurationStats.csv')):
    duration_df.to_csv(os.path.join(output_path, 'DurationStats.csv'), header=True, index=False)
if not os.path.exists(os.path.join(output_path, 'LengthStats.csv')):
    length_df.to_csv(os.path.join(output_path, 'LengthStats.csv'), header=True, index=False)
if not os.path.exists(os.path.join(output_path, 'SpeedStats.csv')):
    speed_df.to_csv(os.path.join(output_path, 'SpeedStats.csv'), header=True, index=False)
if not os.path.exists(os.path.join(output_path, 'VelocityStats.csv')):
    velocity_df.to_csv(os.path.join(output_path, 'VelocityStats.csv'), header=True, index=False)
if not os.path.exists(os.path.join(output_path, 'AccelerationStats.csv')):
    acceleration_df.to_csv(os.path.join(output_path, 'AccelerationStats.csv'), header=True, index=False)
if not os.path.exists(os.path.join(output_path, 'AngleStats.csv')):
    angle_df.to_csv(os.path.join(output_path, 'AngleStats.csv'), header=True, index=False)
if not os.path.exists(os.path.join(output_path, 'CurvatureStats.csv')):
    curvature_df.to_csv(os.path.join(output_path, 'CurvatureStats.csv'), header=True, index=False)

C:\Users\katie\AppData\Local\Temp\ipykernel_43660\51594227.py:75: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df["SubID"] = int(subID)
C:\Users\katie\AppData\Local\Temp\ipykernel_43660\51594227.py:76: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df["SessionID"] = int(sessionID)
C:\Users\katie\AppData\Local\Temp\ipykernel_43660\51594227.py:75: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instea